In [1]:
import glob
import json
from os.path import dirname

import numpy as np

ood_path = "../../logs/exp001/exp001a/sweep-2026-04-28_17-42-26_ood_test_predictions/"

ood_labels_glob = f"{ood_path}/*/labels.npz"
ood_preds_glob = f"{ood_path}/*/predictions.npz"
ood_labels_files = sorted(glob.glob(ood_labels_glob), key=lambda x: int(x.split("/")[-2]))
ood_preds_files = sorted(glob.glob(ood_preds_glob), key=lambda x: int(x.split("/")[-2]))

files_per_run_per_sr = {}
for label_file, pred_file in zip(ood_labels_files, ood_preds_files):
    with open(dirname(label_file) + "/predict-high-freq.log") as f:
        lines = f.readlines()
        sr_line = [line for line in lines if "sleep_stage_frequency" in line][0]
        sleep_stage_sr = int(sr_line.split("=")[1])
        model_run = [line for line in lines if "model.path=" in line][0]
        model_run = model_run.split("=")[1].strip()
    if model_run not in files_per_run_per_sr:
        files_per_run_per_sr[model_run] = {}
    files_per_run_per_sr[model_run][sleep_stage_sr] = (label_file, pred_file)

In [2]:
def total_stage_duration(hypnogram, stage, epoch_length_sec):
    n_epochs = np.sum(hypnogram == stage)
    return n_epochs * epoch_length_sec / 60.0


total_dur_map = {}
total_dur_gt = {}
for sleep_stage in range(5):
    total_dur_map[sleep_stage] = {}
    total_dur_gt[sleep_stage] = {}
    for model_run, files_per_sr in files_per_run_per_sr.items():
        print(f"Model: {model_run}")
        total_dur_map[sleep_stage][model_run] = {}
        for sleep_stage_sr in files_per_sr.keys():
            total_dur_map[sleep_stage][model_run][sleep_stage_sr] = {}
            sr_data = np.load(files_per_sr[sleep_stage_sr][1])
            sr_data_gt = np.load(files_per_run_per_sr[model_run][1][0])
            for s_id in sr_data:
                gt_ss = sr_data_gt[s_id]
                pred_ss = sr_data[s_id]
                gt_art_mask = gt_ss == 9
                if s_id not in total_dur_gt:
                    tsd = total_stage_duration(gt_ss, sleep_stage, 30)
                    total_dur_gt[sleep_stage][s_id] = tsd

                # mark GT artifacts as such in the prediction to remove potential biases
                if sleep_stage_sr == 1:
                    pred_ss[gt_art_mask] = 9
                tsd = total_stage_duration(pred_ss, sleep_stage, 30 / sleep_stage_sr)
                total_dur_map[sleep_stage][model_run][sleep_stage_sr][s_id] = tsd

Model: usleep-run1.pth
Model: usleep-run2.pth
Model: usleep-run3.pth
Model: usleep-run1.pth
Model: usleep-run2.pth
Model: usleep-run3.pth
Model: usleep-run1.pth
Model: usleep-run2.pth
Model: usleep-run3.pth
Model: usleep-run1.pth
Model: usleep-run2.pth
Model: usleep-run3.pth
Model: usleep-run1.pth
Model: usleep-run2.pth
Model: usleep-run3.pth


In [3]:
# save all metrics to file
all_metrics = {
    "total_dur_map": total_dur_map,
    "total_dur_gt": total_dur_gt,
}
with open("table_s4a_usleep.json", "w") as f:
    json.dump(all_metrics, f)